# 중간 프로젝트 - Wildberries 크롤러
# Wildberries Skin Care > Facial Care > Cream

## 1. 상품 정보 크롤링

In [2]:
# Step 1. 필요한 라이브러리 로딩
# !pip install undetected-chromedriver
# !pip install --upgrade undetected-chromedriver
import pandas as pd
import re
import requests
from io import BytesIO
from PIL import Image as PILImage
import openpyxl
from openpyxl.drawing.image import Image as XLImage
from bs4 import BeautifulSoup
import os, time
import undetected_chromedriver as uc
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import random

In [3]:
# Step 2. 사용자에게 카테고리를 입력받기
print("=" * 80)
print(" Wildberries의 Best Skincare 상품 정보 추출하기 ")
print("=" * 80)

site         = 'Wildberries'  # Home > Beauty > Skin Care > Facial Care
FacialCare_url = 'https://www.wildberries.ru/catalog/krasota/uhod-za-kozhey/uhod-za-litsom'

# ── 카테고리 매핑 ──────────────────────────────────────────────────────────────
# 영문 저장명 (파일명 등에 사용)
category_names = {
    '1': 'SkincareSet',
    '2': 'Cream',
    '3': 'Serum',
    '4': 'FaceMask',
    '5': 'Foam',
    '6': 'Toner',
    '7': 'SheetMask',
    '8': 'Lotion'
}

# 사이트 필터에서 사용되는 러시아어 카테고리명 (대소문자 일치 필요)
category_filters = {
    '1': 'Косметический набор для ухода',
    '2': 'Крем',
    '3': 'Сыворотка',
    '4': 'Маска косметическая',
    '5': 'Пенка',
    '6': 'Тоник',
    '7': 'Маска тканевая косметическая',
    '8': 'Лосьон'
}

category = input('''
    1. 스킨케어 세트
    2. 크림
    3. 세럼
    4. 페이스 마스크
    5. 클렌징 폼
    6. 토너
    7. 시트 마스크
    8. 로션
    위 항목 중에서 데이터를 수집할 카테고리 번호를 입력하세요 (예: 2): ''').strip()

if category not in category_names:
    raise ValueError(f"{category}(은)는 유효하지 않은 카테고리입니다. 셀을 다시 실행하세요.")

category_name   = category_names[category]          # 파일명용 영문명
category_filter     = category_filters[category]         # 필터 클릭용 러시아어명

print(f"수집할 제품 카테고리 유형: {category_name} ({category_filter})")

# 현재 날짜를 'YYMMDD' 형식으로 변환
s = time.strftime('%y%m%d', time.localtime())

# 저장 경로 설정
f_dir = input("저장 경로 (Enter: 기본 경로 사용): ")
if f_dir == '':
    f_dir = r"C:\Users\itwill\workspace\Project\중간 프로젝트"
elif f_dir == '1':
    f_dir = r"D:\workspace"

base_name = f'{s}_Ru_{site}_{category_name}'
folder    = os.path.join(f_dir, base_name)
img_dir   = os.path.join(folder, 'images')  # 이미지 저장 전용 하위 폴더

os.makedirs(img_dir, exist_ok=True)
os.chdir(folder)

fc_name = os.path.join(folder, base_name + '.csv')
fx_name = os.path.join(folder, base_name + '.xlsx')

 Wildberries의 Best Skincare 상품 정보 추출하기 
수집할 제품 카테고리 유형: Cream (Крем)


In [4]:
# Step 3. undetected-chromedriver를 이용하여 URL 주소로 접속하기
driver = uc.Chrome(version_main=147)  # 크롬 버전 강제 지정

driver.get(FacialCare_url)
WebDriverWait(driver, 60).until(
    EC.presence_of_element_located((By.CSS_SELECTOR, 'span.goods-count'))
)  # 상품 수 요소가 로딩될 때까지 최대 60초 대기
driver.maximize_window()
time.sleep(random.uniform(1, 3))

#### Facial Care 카테고리에서 'Категория' 필터로 제품 종류 선택하기

> косметический набор для ухода 스킨케어 세트 / крем 크림 / сыворотка 세럼
>
> маска косметическая 페이스 마스크 / пенка 클렌징 폼 / тоник 토너
>
> маска тканевая косметическая 시트 마스크 / лосьон 로션
>
> Показать все 모두 보기 / Сбросить 초기화 / Готово 완료

In [5]:
# Step 4. 'Категория' 필터 클릭 → Step 2에서 선택한 카테고리 적용 → 수집 건수 입력

# ── 4-1. 카테고리 필터 적용 ────────────────────────────────────────────────────

try:
    # 'Категория' 필터 버튼 클릭
    btn = WebDriverWait(driver, 10).until(
        EC.element_to_be_clickable((
            By.CSS_SELECTOR,
            "div[data-testid='catalog-filter'] button.dropdown-filter__btn--xsubject"
        ))
    )
    btn.click()
    time.sleep(random.uniform(1, 2))

    # 카테고리 체크박스 클릭
    item = WebDriverWait(driver, 10).until(
        EC.presence_of_element_located((
            By.XPATH,
            f"//span[contains(@class, 'checkbox-with-text__text')]"
            f"[normalize-space(translate(., '\u00a0 ', ''))='{category_filter}']"
            f"/ancestor::div[contains(@class, 'j-list-item')][1]"
        ))
    )
    driver.execute_script("arguments[0].click();", item)
    time.sleep(random.uniform(0.5, 1))

    # ✅ 수정: filter-btn__main 으로 presence 먼저 확인 후 JS 클릭
    # element_to_be_clickable 대신 presence_of_element_located 사용
    done_btn = WebDriverWait(driver, 10).until(
        EC.presence_of_element_located((
            By.CSS_SELECTOR, "button.filter-btn__main"
        ))
    )
    driver.execute_script("arguments[0].scrollIntoView(true);", done_btn)
    time.sleep(0.3)
    driver.execute_script("arguments[0].click();", done_btn)
    time.sleep(random.uniform(2, 3))
    print(f"카테고리 필터 적용 완료: {category_filter}")

except Exception as e:
    print(f"카테고리 필터 적용 실패: {e}")

    # ✅ 폴백: XPath로 버튼 텍스트 직접 검색
    try:
        done_btn = WebDriverWait(driver, 5).until(
            EC.presence_of_element_located((
                By.XPATH,
                "//button[contains(@class, 'filter-btn__main') and contains(text(), 'Готово')]"
            ))
        )
        driver.execute_script("arguments[0].click();", done_btn)
        time.sleep(random.uniform(2, 3))
        print(f"카테고리 필터 적용 완료 (폴백): {category_filter}")
    except Exception as e2:
        print(f"폴백도 실패: {e2}")
        print("필터 없이 전체 Facial Care 상품으로 진행합니다.")

# ── 4-2. 필터 적용 후 전체 상품 수 확인 및 수집 건수 입력 ─────────────────────
WebDriverWait(driver, 15).until(
    EC.presence_of_element_located((By.CSS_SELECTOR, 'span.goods-count'))
)
html  = driver.page_source
soup  = BeautifulSoup(html, 'html.parser')

category_title = soup.select_one('h1.catalog-title').get_text(strip=True)
total_cnt      = soup.select_one('span.goods-count').get_text().replace('товаров', '').strip()

print(f"{category_title} > {category_filter} 카테고리에 {total_cnt}개의 상품이 등록되어 있습니다.")

# 수집 건수 입력 (검색 결과는 인기순이므로 입력된 수만큼 상위 상품 수집)
cnt = int(input(f"상위 몇 건을 수집하시겠습니까? ({total_cnt}건 이하 입력): "))
print(f"인기순 상위 {cnt}건을 수집하겠습니다.")

카테고리 필터 적용 완료: Крем
Косметика по уходу за лицом > Крем 카테고리에 81 741 товар개의 상품이 등록되어 있습니다.
인기순 상위 100건을 수집하겠습니다.


In [6]:
# Step 5. 사용자 정의 함수

def find_value_by_key(soup_obj, key_text):
    """th 텍스트로 원하는 tr을 찾아 td 값을 반환하는 함수"""
    for tr in soup_obj.select('tr'):
        th = tr.select_one('th.cellKey--eGe6N')
        if th and key_text.lower() in th.get_text(strip=True).lower():
            td = tr.select_one('td.cellValue--hHBJB div')
            return td.get_text(strip=True) if td else None
    return None


def download_image(img_url, save_dir, filename):
    """img_url을 받아 100×100으로 리사이즈 후 로컬 저장, 저장 경로 반환"""
    try:
        headers  = {'User-Agent': 'Mozilla/5.0'}  # 차단 방지용 헤더
        response = requests.get(img_url, headers=headers, timeout=10)
        response.raise_for_status()
        img_data = PILImage.open(BytesIO(response.content)).convert('RGB')
        img_data = img_data.resize((100, 100))   # 셀에 맞게 100×100으로 리사이즈
        save_path = os.path.join(save_dir, filename)
        img_data.save(save_path, 'JPEG')
        return save_path
    except Exception as e:
        print(f'이미지 다운로드 실패: {e}')
        return None

In [7]:
# Step 6. 상품 링크 수집 (lazy loading 대응: 스크롤하며 cnt개 이상 링크 확보)

# 수집 데이터를 담을 리스트
ranking2  = []   # 1. 인기 순위 (1위부터)
img2      = []   # 2. 제품 이미지 URL
prd2      = []   # 3. 제품 이름
brd2      = []   # 4. 브랜드 이름
price2    = []   # 5. 판매 가격
rating2   = []   # 6. 별점
rev_cnt2  = []   # 7. 리뷰 수
rev_url2  = []   # 12. 리뷰 페이지 링크
ci2       = []   # 8. 화장품 성분
benefits2 = []   # 9. 기능
skin_tp2  = []   # 10. 피부 타입
url2      = []   # 11. 상품 링크

prd_links       = []       # 개별 상품 링크
collected_hrefs = set()    # 중복 방지

while len(prd_links) < cnt:
    soup      = BeautifulSoup(driver.page_source, 'html.parser')
    prd_items = soup.select('div.product-card-list article.product-card')

    new_found = False
    for article in prd_items:
        link_tag = article.select_one('a.product-card__link.j-card-link')
        href     = link_tag['href'] if link_tag else None
        if href and href not in collected_hrefs:
            collected_hrefs.add(href)
            prd_links.append(href)
            new_found = True
        if len(prd_links) >= cnt:
            break

    if len(prd_links) >= cnt:
        break

    # 새 상품이 발견되지 않으면 스크롤 후 새 항목 로딩 대기
    if not new_found:
        prev_count = len(driver.find_elements(By.CSS_SELECTOR, 'article.product-card'))
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        try:
            WebDriverWait(driver, 10).until(
                lambda d: len(d.find_elements(By.CSS_SELECTOR, 'article.product-card')) > prev_count
            )
        except:
            print("더 이상 로딩되는 상품이 없습니다. 수집을 종료합니다.")
            break

print(f"총 {len(prd_links)}개의 상품 링크를 수집했습니다.")

총 100개의 상품 링크를 수집했습니다.


In [8]:
# Step 7. 상품 상세 정보 수집
# 검색 결과가 인기순이므로 ranking은 1위부터 순서대로 부여

count        = 1
failed_links = []   # (리스트 인덱스, prd_link) — 7-2에서 재시도할 상품

for prd_link in prd_links:
    if count > cnt:
        break

    print(f"\n{count}번째 상품 데이터 추출 중...\n")
    current_idx = len(url2)   # 이번 상품이 리스트에 추가될 인덱스 위치

    try:
        driver.get(prd_link)
        WebDriverWait(driver, 10).until(
            lambda d: (
                len(d.find_elements(By.CSS_SELECTOR, 'span[class*="btnDetailText"]')) > 0
                and len(d.find_elements(By.CSS_SELECTOR, 'span[class*="productHeaderBrandText"]')) > 0
            )
        )
        detail_soup = BeautifulSoup(driver.page_source, 'html.parser')

        # 1. 제품 순위 (인기순 1위부터 순서대로, 1-base)
        ranking = count
        print(f'Ranking: {ranking}')
        ranking2.append(ranking)

        # 2. 제품 대표 이미지 URL 추출
        img_url = 'Null'
        try:
            img_tag = detail_soup.select_one('img[data-src-pb]')
            if img_tag and img_tag.get('src'):
                raw     = img_tag['src']
                img_url = 'https:' + raw if raw.startswith('//') else raw
        except:
            img_url = 'Null'
        print(f'Image URL: {img_url}')
        img2.append(img_url)

        # 3. 제품 이름 추출
        prd = 'Null'
        try:
            prd = detail_soup.select_one('h2[class*="productTitle--lfc4o"]').get_text(strip=True)
        except:
            prd = 'Null'
        print(f'Product: {prd}')
        prd2.append(prd)

        # 4. 브랜드 이름 추출
        brd = 'Null'
        try:
            brd_tag = detail_soup.select_one('span[class*="productHeaderBrandText"]')
            brd     = brd_tag.get_text(strip=True) if brd_tag else 'Null'
        except:
            brd = 'Null'
        print(f'Brand: {brd}')
        brd2.append(brd)

        # 5. 판매가 추출 (할인가 우선, 없으면 정가)
        price = 0
        try:
            price_tag = detail_soup.select_one('h2.mo-typography_color_danger, h2.mo-typography_color_accent')
            price     = int(re.sub(r'\D', '', price_tag.get_text()))
        except:
            price = 0
        print(f'Price: {price}')
        price2.append(price)

        # 6. 별점 추출
        rating = 0.0
        try:
            star_tag = detail_soup.select_one('span[class*="productReviewRating"]')
            if star_tag:
                match = re.search(r'\d+[,.]\d+|\d+', star_tag.get_text(strip=True))
                if match:
                    rating = float(match.group().replace(',', '.'))
        except:
            rating = 0.0
        print(f'Rating: {rating}')
        rating2.append(rating)

        # 7. 리뷰 수 추출
        rev_cnt = 0
        try:
            star_tag = detail_soup.select_one('span[class*="productReviewRating"]')
            if star_tag:
                review = star_tag.get_text(strip=True)
                match2 = re.search(r'·\s*([\d\s\u00a0]+)\s*[а-яА-Я]', review)
                if match2:
                    rev_cnt = int(re.sub(r'[\s\u00a0]', '', match2.group(1).strip()))
        except:
            rev_cnt = 0
        print(f'Review_Cnt: {rev_cnt}')
        rev_cnt2.append(rev_cnt)

        # 8. 리뷰 페이지 링크 수집
        rev_url = 'Null'
        try:
            # href에 'feedbacks'가 포함된 <a> 태그로 선택 (클래스 해시 무관)
            rev_tag = detail_soup.select_one('a[href*="feedbacks"]')
            if rev_tag and rev_tag.get('href'):
                rev_url = 'https://www.wildberries.ru' + rev_tag['href']
        except:
            rev_url = 'Null'
        print(f'Review_Page: {rev_url}')
        rev_url2.append(rev_url)

        # 상품 정보 더보기 버튼 클릭
        try:
            driver.find_element(By.CSS_SELECTOR, 'span[class*="btnDetailText"]').click()
            time.sleep(random.uniform(1, 2))
            detail_soup = BeautifulSoup(driver.page_source, 'html.parser')
        except:
            pass

        # 9. 화장품 성분(Состав) 추출
        ci = 'Null'
        try:
            ci_raw  = find_value_by_key(detail_soup, 'Состав')
            ci_list = [item.strip() for item in ci_raw.split(';') if item.strip()]
            ci      = ', '.join(ci_list)
        except:
            ci = 'Null'
        print(f'Ingredients: {ci}')
        ci2.append(ci)

        # 10. 화장품 기능(Действие) 추출
        benefits = 'Null'
        try:
            benefits_raw  = find_value_by_key(detail_soup, 'Действие')
            benefits_list = [item.strip() for item in benefits_raw.split(';') if item.strip()]
            benefits      = ', '.join(benefits_list)
        except:
            benefits = 'Null'
        print(f'Benefits: {benefits}')
        benefits2.append(benefits)

        # 11. 피부 타입(Тип кожи) 추출
        skin_tp = 'Null'
        try:
            skin_tp = find_value_by_key(detail_soup, 'Тип кожи') or 'Null'
        except:
            skin_tp = 'Null'
        print(f'Skin_Type: {skin_tp}')
        skin_tp2.append(skin_tp)

        # 12. 상품 링크
        url2.append(prd_link)

    # 오류 처리 — Null/0으로 채우고 failed_links에 기록 (7-2에서 재시도)
    except Exception as e:
        print(f'Error: {e}')
        print(f'→ failed_links에 추가됩니다. (인덱스: {current_idx})')
        ranking2.append(count)   # 순위는 실패해도 현재 순서 유지
        for num_list in [price2, rating2, rev_cnt2]:
            num_list.append(0)
        for str_list in [img2, prd2, brd2, rev_url2, ci2, benefits2, skin_tp2]:
            str_list.append('Null')
        url2.append(prd_link)   # 링크는 기록 유지
        failed_links.append((current_idx, prd_link))   # 재시도 대상 등록
        time.sleep(2)

    count += 1

print("=" * 50)
print(f"데이터 수집 완료 (성공: {len(prd_links) - len(failed_links)}건 / 재시도 대상: {len(failed_links)}건)")


1번째 상품 데이터 추출 중...

Ranking: 1
Image URL: https://basket-01.wbbasket.ru/vol48/part4803/4803233/images/tm/1.webp
Product: Крем для лица мужской увлажняющий, 75 мл
Brand: NIVEA
Price: 294
Rating: 4.9
Review_Cnt: 34980
Review_Page: https://www.wildberries.ru/catalog/4803233/feedbacks?size=17665899
Ingredients: основа косметическая
Benefits: восстановление
Skin_Type: для всех типов кожи

2번째 상품 데이터 추출 중...

Ranking: 2
Image URL: https://basket-38.wbbasket.ru/vol8639/part863944/863944820/images/tm/1.webp
Product: Солнцезащитный крем для лица и тела увлажняющий спф 50 Spf
Brand: Levape
Price: 491
Rating: 5.0
Review_Cnt: 648
Review_Page: https://www.wildberries.ru/catalog/863944820/feedbacks?size=1300984330
Ingredients: aqua, Octyl methoxy cinnamate, homosalate, Hydrolyzed Collagen, vitamin Е, Octyl Salicylate, Benzophenone-4,, ethylhexyl methoxycinnamate, Panthenol,, гиалуроновая кислота
Benefits: солнцезащитное, защита, для загара
Skin_Type: для всех типов кожи

3번째 상품 데이터 추출 중...

Ranking

In [9]:
# Step 7-2. 브랜드 요소가 없어 오류가 났던 상품 재수집 (브랜드 로딩 대기 없이)

if failed_links:
    print(f"총 {len(failed_links)}건 재시도 시작...")

    for idx, prd_link in failed_links:
        print(f"\n재시도 중... (원본 인덱스: {idx}, URL: {prd_link})\n")

        try:
            driver.get(prd_link)

            # 브랜드 대기 조건 제외 — '더보기' 버튼만 대기
            WebDriverWait(driver, 20).until(
                lambda d: len(d.find_elements(By.CSS_SELECTOR, 'span[class*="btnDetailText"]')) > 0
            )
            detail_soup = BeautifulSoup(driver.page_source, 'html.parser')

            # 1. 제품 순위 — idx 기반으로 기존 리스트 덮어쓰기 (순위는 이미 count로 할당됨)
            # ranking2[idx]는 이미 count 값으로 설정되어 있으므로 그대로 유지

            # 2. 제품 대표 이미지 URL 추출
            img_url = 'Null'
            try:
                img_tag = detail_soup.select_one('img[data-src-pb]')
                if img_tag and img_tag.get('src'):
                    raw     = img_tag['src']
                    img_url = 'https:' + raw if raw.startswith('//') else raw
            except:
                img_url = 'Null'
            print(f'Image URL: {img_url}')
            img2[idx] = img_url

            # 3. 제품 이름 추출
            prd = 'Null'
            try:
                prd = detail_soup.select_one('h2[class*="productTitle--lfc4o"]').get_text(strip=True)
            except:
                prd = 'Null'
            print(f'Product: {prd}')
            prd2[idx] = prd

            # 4. 브랜드 (없으면 Null — 대기 없이 그냥 추출)
            brd = 'Null'
            try:
                brd_tag = detail_soup.select_one('span[class*="productHeaderBrandText"]')
                brd     = brd_tag.get_text(strip=True) if brd_tag else 'Null'
            except:
                brd = 'Null'
            print(f'Brand: {brd}')
            brd2[idx] = brd

            # 5. 판매가 추출 (할인가 우선, 없으면 정가)
            price = 0
            try:
                price_tag = detail_soup.select_one('h2.mo-typography_color_danger, h2.mo-typography_color_accent')
                price     = int(re.sub(r'\D', '', price_tag.get_text()))
            except:
                price = 0
            print(f'Price: {price}')
            price2[idx] = price

            # 6. 별점 추출
            rating = 0.0
            try:
                star_tag = detail_soup.select_one('span[class*="productReviewRating"]')
                if star_tag:
                    match = re.search(r'\d+[,.]\d+|\d+', star_tag.get_text(strip=True))
                    if match:
                        rating = float(match.group().replace(',', '.'))
            except:
                rating = 0.0
            print(f'Rating: {rating}')
            rating2[idx] = rating

            # 7. 리뷰 수 추출
            rev_cnt = 0
            try:
                star_tag = detail_soup.select_one('span[class*="productReviewRating"]')
                if star_tag:
                    review = star_tag.get_text(strip=True)
                    match2 = re.search(r'·\s*([\d\s\u00a0]+)\s*[а-яА-Я]', review)
                    if match2:
                        rev_cnt = int(re.sub(r'[\s\u00a0]', '', match2.group(1).strip()))
            except:
                rev_cnt = 0
            print(f'Review_Cnt: {rev_cnt}')
            rev_cnt2[idx] = rev_cnt

            # 8. 리뷰 페이지 링크 수집
            rev_url = 'Null'
            try:
                # href에 'feedbacks'가 포함된 <a> 태그로 선택 (클래스 해시 무관)
                rev_tag = detail_soup.select_one('a[href*="feedbacks"]')
                if rev_tag and rev_tag.get('href'):
                    rev_url = 'https://www.wildberries.ru' + rev_tag['href']
            except:
                rev_url = 'Null'
            print(f'Review_Page: {rev_url}')
            rev_url2[idx] = rev_url

            # 상품 정보 더보기 버튼 클릭
            try:
                driver.find_element(By.CSS_SELECTOR, 'span[class*="btnDetailText"]').click()
                time.sleep(random.uniform(1, 2))
                detail_soup = BeautifulSoup(driver.page_source, 'html.parser')
            except:
                pass

            # 9. 화장품 성분(Состав) 추출
            ci = 'Null'
            try:
                ci_raw  = find_value_by_key(detail_soup, 'Состав')
                ci_list = [item.strip() for item in ci_raw.split(';') if item.strip()]
                ci      = ', '.join(ci_list)
            except:
                ci = 'Null'
            print(f'Ingredients: {ci}')
            ci2[idx] = ci

            # 10. 화장품 기능(Действие) 추출
            benefits = 'Null'
            try:
                benefits_raw  = find_value_by_key(detail_soup, 'Действие')
                benefits_list = [item.strip() for item in benefits_raw.split(';') if item.strip()]
                benefits      = ', '.join(benefits_list)
            except:
                benefits = 'Null'
            print(f'Benefits: {benefits}')
            benefits2[idx] = benefits

            # 11. 피부 타입(Тип кожи) 추출
            skin_tp = 'Null'
            try:
                skin_tp = find_value_by_key(detail_soup, 'Тип кожи') or 'Null'
            except:
                skin_tp = 'Null'
            print(f'Skin_Type: {skin_tp}')
            skin_tp2[idx] = skin_tp

            print(f'→ 재수집 성공 (인덱스: {idx})')

        except Exception as e:
            print(f'→ 재수집도 실패: {e} (인덱스: {idx}, URL: {prd_link})')
            # 이미 Null/0으로 채워져 있으므로 추가 처리 불필요
            time.sleep(2)

    print("=" * 50)
    print("Step 7-2 재수집 완료")
else:
    print("재시도 대상 없음 — 모든 상품이 Step 7에서 정상 수집되었습니다.")

총 2건 재시도 시작...

재시도 중... (원본 인덱스: 45, URL: https://www.wildberries.ru/catalog/1002142412/detail.aspx)

Image URL: https://basket-41.wbbasket.ru/vol10021/part1002142/1002142412/images/tm/1.webp
Product: Крем лифтинг амвей
Brand: Null
Price: 6307
Rating: 0.0
Review_Cnt: 0
Review_Page: https://www.wildberries.ru/catalog/1002142412/feedbacks?imtId=1817967065&size=1498098623
Ingredients: Пептиды,семян белой чиа
Benefits: лифтинг, упругость, эластичность
Skin_Type: для всех типов кожи
→ 재수집 성공 (인덱스: 45)

재시도 중... (원본 인덱스: 63, URL: https://www.wildberries.ru/catalog/174390880/detail.aspx)

Image URL: https://basket-12.wbbasket.ru/vol1743/part174390/174390880/images/tm/1.webp
Product: Увлажняющий крем для лица с азиатской центеллой skin1004
Brand: Null
Price: 1254
Rating: 4.8
Review_Cnt: 814
Review_Page: https://www.wildberries.ru/catalog/174390880/feedbacks?size=289147987
Ingredients: гиалуроновая кислота, центелла азиатская, керамиды, лецитин, церамиды, Транексамовая кислота, камфора, Витами

In [9]:
# Step 7 결과 보존하는 백업 리스트
ranking2_raw  = ranking2.copy()
img2_raw      = img2.copy()
prd2_raw      = prd2.copy()
brd2_raw      = brd2.copy()
price2_raw    = price2.copy()
rating2_raw   = rating2.copy()
rev_cnt2_raw  = rev_cnt2.copy()
ci2_raw       = ci2.copy()
benefits2_raw = benefits2.copy()
skin_tp2_raw  = skin_tp2.copy()
url2_raw      = url2.copy()
rev_url2_raw  = rev_url2.copy()

print(f"백업 완료: {len(ranking2_raw)}개")
print(f"ranking2_raw: {ranking2_raw}")

백업 완료: 20개
ranking2_raw: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20]


### 수집한 20건 상품 필터링: 리뷰 수 200개 이상이면서 인기순 상위 10개 상품만 남기기

In [10]:
# Step 8. rev_cnt 200개 이상 & Ranking 상위 10개 제품 필터링
# 항상 원본 백업에서 복원 (중복 실행 방지)
ranking2  = ranking2_raw.copy()
img2      = img2_raw.copy()
prd2      = prd2_raw.copy()
brd2      = brd2_raw.copy()
price2    = price2_raw.copy()
rating2   = rating2_raw.copy()
rev_cnt2  = rev_cnt2_raw.copy()
ci2       = ci2_raw.copy()
benefits2 = benefits2_raw.copy()
skin_tp2  = skin_tp2_raw.copy()
url2      = url2_raw.copy()
rev_url2  = rev_url2_raw.copy()

# ── 8-1. 임시 DataFrame 생성 (rev_url2 포함) ───────────────────────────────
temp_df = pd.DataFrame({
    'Ranking'     : pd.Series(ranking2),
    'Image'       : pd.Series(img2),
    'Product_Name': pd.Series(prd2),
    'Brand'       : pd.Series(brd2),
    'Price(₽)'    : pd.Series(price2),
    'Rating'      : pd.Series(rating2),
    'Review_Cnt'  : pd.Series(rev_cnt2),
    'Ingredients' : pd.Series(ci2),
    'Benefits'    : pd.Series(benefits2),
    'Skin_Type'   : pd.Series(skin_tp2),
    'URL'         : pd.Series(url2),
    'Review_URL'     : pd.Series(rev_url2)    
})
print(f"필터링 전 원본:{len(temp_df)}개")
display(temp_df[['Ranking', 'Brand','Review_URL']])

# ── 8-2. 필터링 ────────────────────────────────────────────────────────────
temp_df['Review_Cnt'] = pd.to_numeric(temp_df['Review_Cnt'], errors='coerce').fillna(0).astype(int)
temp_df['Ranking']    = pd.to_numeric(temp_df['Ranking'],    errors='coerce').fillna(0).astype(int)

filtered_df = (
    temp_df[temp_df['Review_Cnt'] >= 200]
    .sort_values('Ranking')
    .head(10)
    .reset_index(drop=True)  
)
      
print(f"→ 필터링 후: {len(filtered_df)}개 (리뷰 200↑, Ranking 상위 10)")
print(filtered_df[['Ranking', 'Brand', 'Review_Cnt', 'Review_URL']].to_string(index=False))

# ── 8-3. 리스트 변수 재할당 ────────────────────────────────────────────────
ranking2  = filtered_df['Ranking'].tolist()
img2      = filtered_df['Image'].tolist()
prd2      = filtered_df['Product_Name'].tolist()
brd2      = filtered_df['Brand'].tolist()
price2    = filtered_df['Price(₽)'].tolist()
rating2   = filtered_df['Rating'].tolist()
rev_cnt2  = filtered_df['Review_Cnt'].tolist()
ci2       = filtered_df['Ingredients'].tolist()
benefits2 = filtered_df['Benefits'].tolist()
skin_tp2  = filtered_df['Skin_Type'].tolist()
url2      = filtered_df['URL'].tolist()
rev_url2  = filtered_df['Review_URL'].tolist()  

필터링 전 원본:20개


,Ranking,Brand,Review_URL
0,1,Kottur,https://www.wildberries.ru/catalog/493818806/f...
1,2,Гельтек,https://www.wildberries.ru/catalog/624803241/f...
2,3,IRIS COSMETIC,https://www.wildberries.ru/catalog/172282016/f...
3,4,LABÁ,https://www.wildberries.ru/catalog/90002926/fe...
4,5,SADOER,https://www.wildberries.ru/catalog/461393225/f...
5,6,FABRIK Cosmetology,https://www.wildberries.ru/catalog/258293575/f...
6,7,SAVONRY,https://www.wildberries.ru/catalog/893959602/f...
7,8,Compliment,https://www.wildberries.ru/catalog/378889076/f...
8,9,WEIS,https://www.wildberries.ru/catalog/260889862/f...
9,10,Levape,https://www.wildberries.ru/catalog/863074092/f...


→ 필터링 후: 10개 (리뷰 200↑, Ranking 상위 10)
 Ranking              Brand  Review_Cnt                                                             Review_URL
       1             Kottur      145564  https://www.wildberries.ru/catalog/493818806/feedbacks?size=687154703
       2            Гельтек        1050  https://www.wildberries.ru/catalog/624803241/feedbacks?size=847037607
       3      IRIS COSMETIC        6194  https://www.wildberries.ru/catalog/172282016/feedbacks?size=285878653
       4               LABÁ       18339   https://www.wildberries.ru/catalog/90002926/feedbacks?size=145011801
       5             SADOER        1182  https://www.wildberries.ru/catalog/461393225/feedbacks?size=649393160
       6 FABRIK Cosmetology        1487  https://www.wildberries.ru/catalog/258293575/feedbacks?size=401925959
       8         Compliment        1317  https://www.wildberries.ru/catalog/378889076/feedbacks?size=551527551
       9               WEIS         645  https://www.wildberries.ru/catalo

In [11]:
# 디버깅: 각 리스트 길이 확인
print(f"rev_url2  : {len(rev_url2)}")
print(f"ranking2  : {len(ranking2)}")
print(f"prd2      : {len(prd2)}")
print(f"brd2      : {len(brd2)}")

rev_url2  : 10
ranking2  : 10
prd2      : 10
brd2      : 10


### 상품 정보 데이터: CSV 파일 및 엑셀 파일로 저장하기

In [10]:
# Step 9. 상품 정보 데이터를 CSV & xlsx 파일로 저장

# ── 9-1. DataFrame 생성 ──────────────────────────────────────────────────────
wildberries = pd.DataFrame({
    'Ranking'    : pd.Series(ranking2),
    'Image'      : pd.Series(img2),       # csv에는 URL 저장
    'Product_Name': pd.Series(prd2),
    'Brand'      : pd.Series(brd2),
    'Price(₽)'   : pd.Series(price2),
    'Rating'     : pd.Series(rating2),
    'Review_Cnt' : pd.Series(rev_cnt2),
    'Ingredients': pd.Series(ci2),
    'Benefits'   : pd.Series(benefits2),
    'Skin_Type'  : pd.Series(skin_tp2),
    'URL'        : pd.Series(url2),
    'Review_URL' : pd.Series(rev_url2)
})

# ── 9-2. CSV 저장 ─────────────────────────────────────────────────────────────
wildberries.to_csv(fc_name, encoding='utf-8-sig', index=False)
print(f"CSV 저장 완료: {fc_name}")

# ── 8-3. xlsx 저장 (Image 컬럼에 이미지 직접 삽입) ───────────────────────────
wildberries_xl = wildberries.copy()
wildberries_xl['Image'] = ''   # 이미지 자리 확보 (URL 대신 빈 칸)
wildberries_xl.to_excel(fx_name, index=False, engine='openpyxl')

# openpyxl로 파일을 다시 열어 이미지 삽입
wb = openpyxl.load_workbook(fx_name)
ws = wb.active

CELL_PX   = 100   # 이미지 크기 (픽셀)
ROW_START = 2     # 헤더(1행) 제외, 데이터는 2행부터

ws.column_dimensions['A'].width = 16   # A열(Ranking) 너비
ws.column_dimensions['B'].width = 16   # B열(Image) 너비

for i, img_url in enumerate(img2):
    row = ROW_START + i
    ws.row_dimensions[row].height = CELL_PX * 0.75   # 행 높이 설정 (pt 단위)

    if img_url and img_url != 'Null':
        img_path = download_image(img_url, img_dir, f'img_{i+1}.jpg')
        if img_path:
            try:
                xl_img        = XLImage(img_path)
                xl_img.width  = CELL_PX
                xl_img.height = CELL_PX
                ws.add_image(xl_img, f'B{row}')   # B열(Image) 해당 행에 이미지 삽입
            except Exception as e:
                print(f'{row}행 이미지 삽입 실패: {e}')

wb.save(fx_name)
print(f"xlsx 저장 완료 (이미지 포함): {fx_name}")

CSV 저장 완료: C:\Users\itwill\workspace\Project\중간 프로젝트\260512_Ru_Wildberries_Cream\260512_Ru_Wildberries_Cream.csv
xlsx 저장 완료 (이미지 포함): C:\Users\itwill\workspace\Project\중간 프로젝트\260512_Ru_Wildberries_Cream\260512_Ru_Wildberries_Cream.xlsx


In [ ]:
# 생성한 엑셀 파일 정보 확인 & 리뷰 URL 컬럼 불러오기
rev_df = pd.read_excel(f"{fx_name}")
print(rev_df.columns)
print(rev_df.info())
rev_page = rev_df['Review_URL'].tolist() # 각 상품의 리뷰 페이지 url 리스트 -> for 반복문에 사용
print(f"리뷰 페이지 리스트 길이: {len(rev_page)}개")
rev_page

In [ ]:
# Step 12. 요약 정보 출력 및 드라이버 종료

print("\n")
print("=" * 100)
print(f"1. 요청된 총 {cnt}건의 상품 중 실제 크롤링된 상품 수: {len(prd2)}건")
print(f"2. 상품 CSV 파일: {fc_name}")
print(f"3. 상품 xlsx 파일 (이미지 포함): {fx_name}")
print("=" * 100)

driver.quit()